# SemanticPrism Diagnostic Workflow
Step-by-step interactive workflow breaking down the pipeline execution.
Now mathematically 1:1 functionally identical to the `pipeline.py` JSON storage footprint!


In [1]:
import os
import json
import glob
import time
from datetime import datetime
import logging
import IPython.display
import nest_asyncio
nest_asyncio.apply()

from typing import Any
from src.core.logger import save_execution_log
from src.extraction.extractor import ExtractionPipeline
from src.extraction.normalize_text import execute_normalization_phase
from src.embedding.embedding import EmbeddingPipeline
from src.nlp.hypernyms import HypernymPipeline
from src.nlp.nlp_mapping import NamingResolutionPipeline
from src.topology.graph_builder import TopologyEngine
from src.synthesis.synthesizer import SynthesisEngine
from src.helpers.visualizer import SemanticVisualizer

def _save_state(data: Any, filepath: str):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        default_encoder = lambda x: list(x) if isinstance(x, set) else str(x)
        if isinstance(data, list) and len(data) > 0 and hasattr(data[0], 'model_dump'):
            json.dump([item.model_dump(mode='json') for item in data], f, indent=4, default=default_encoder)
        elif hasattr(data, 'model_dump'):
            json.dump(data.model_dump(mode='json'), f, indent=4, default=default_encoder)
        else:
            json.dump(data, f, indent=4, default=default_encoder)

# Dynamically locate text files
target_dir = "inputs/testdocs"
files = glob.glob(os.path.join(target_dir, "*.txt")) + glob.glob(os.path.join(target_dir, "*.md"))
raw_texts = []
for path in files:
    with open(path, "r", encoding="utf-8") as f:
        raw_texts.append(f.read())
print(f"Successfully loaded {len(raw_texts)} documents natively from {target_dir}.")
if not raw_texts:
    print("WARNING: Create physical text files manually inside inputs/testdocs gracefully.")

Successfully loaded 3 documents natively from inputs/testdocs.


### 1. Extraction Pipeline


In [2]:
extractor = ExtractionPipeline("config.yaml")

start_time = time.time()
start_datetime = datetime.now()
workflow_errors = []

master_context = None

def _dump_current_log():
    all_errors = workflow_errors.copy()
    if hasattr(extractor, 'llm'): all_errors.extend(extractor.llm.error_history)
    if 'hypernyms' in globals() and hasattr(hypernyms, 'llm'): all_errors.extend(hypernyms.llm.error_history)
    if 'synthesizer' in globals() and hasattr(synthesizer, 'llm'): all_errors.extend(synthesizer.llm.error_history)
    
    all_ctxs = []
    if hasattr(extractor, 'llm'): all_ctxs.extend(extractor.llm.context_history)
    if 'hypernyms' in globals() and hasattr(hypernyms, 'llm'): all_ctxs.extend(hypernyms.llm.context_history)
    if 'synthesizer' in globals() and hasattr(synthesizer, 'llm'): all_ctxs.extend(synthesizer.llm.context_history)

    distilled_t_count = len(master_context.master_themes) if master_context and hasattr(master_context, 'master_themes') else 0
    
    metrics = {
        "start_datetime": start_datetime,
        "duration": time.time() - start_time,
        "use_async": getattr(extractor, 'use_async', False),
        "model_name": extractor.config.get('llm', {}).get('model_name', 'Unknown'),
        "connection_protocol": extractor.config.get('llm', {}).get('connection_protocol', 'Unknown'),
        "api_backend": extractor.config.get("llm", {}).get("api_backend", "Unknown"),
        "doc_count": len(raw_texts),
        "doc_lengths": [len(doc) for doc in raw_texts],
        "all_ctxs": all_ctxs,
        "all_themes_count": len(all_themes) if 'all_themes' in globals() else 0,
        "distilled_t_count": distilled_t_count,
        "raw_triples_count": len(raw_triples) if 'raw_triples' in globals() else 0,
        "orig_subjs": len(original_subjs) if 'original_subjs' in globals() else 0,
        "orig_preds": len(original_preds) if 'original_preds' in globals() else 0,
        "orig_objs": len(original_objs) if 'original_objs' in globals() else 0,
        "norm_subjs": len(norm_subjs) if 'norm_subjs' in globals() else 0,
        "norm_preds": len(norm_preds) if 'norm_preds' in globals() else 0,
        "norm_objs": len(norm_objs) if 'norm_objs' in globals() else 0,
        "all_errors": all_errors
    }
    workflow_logger = logging.getLogger("DiagnosticWorkflow")
    save_execution_log(metrics, workflow_logger)

# Baseline trackings 

visualizer = SemanticVisualizer()
visualizer.visualize_triples(raw_triples, "outputs/01_extraction/01_raw_triples_graph.html", "Phase 1: Raw Extractions")
IPython.display.display(IPython.display.IFrame("outputs/01_extraction/01_raw_triples_graph.html", width="100%", height="600px"))

2026-04-20 14:44:15 | INFO     | ContextManager | Initialized ContextManager - VRAM Free: 11851 MB, RAM Free: 21275 MB
2026-04-20 14:44:15 | INFO     | ExtractionPipeline | Initialized ExtractionPipeline (Model: google-vertex-model, Use Async: True)
Processing themes for document 1/3
2026-04-20 14:44:15 | INFO     | ExtractionPipeline | Phase 1: Executing Theme Discovery...
Processing themes for document 2/3
2026-04-20 14:44:33 | INFO     | ExtractionPipeline | Phase 1: Executing Theme Discovery...
Processing themes for document 3/3
2026-04-20 14:44:36 | INFO     | ExtractionPipeline | Phase 1: Executing Theme Discovery...
2026-04-20 14:44:41 | INFO     | ExtractionPipeline | Phase 1.5: Consolidating overarching Master Domain.
Processing triples for document 1/3
2026-04-20 14:45:33 | INFO     | ExtractionPipeline | Phase 2: Executing Logical Triple Extraction Explicitly...
2026-04-20 14:45:33 | INFO     | ExtractionPipeline | Extracting triples from logic chunk 1/1...
Processing triple

In [3]:
original_subjs = {t.subject for t in raw_triples}
original_preds = {t.predicate for t in raw_triples}
original_objs = {t.object for t in raw_triples}

norm_subjs, norm_preds, norm_objs = await execute_normalization_phase(
    extractor,
    raw_triples,
    master_domain,
    _save_state
)

2026-04-20 14:51:54 | INFO     | Visualizer | Initializing Static Network Visualizer efficiently rationally.
2026-04-20 14:51:54 | INFO     | Visualizer | Generating abstract visual organically seamlessly correctly optimally for 175 logical facts securely.
2026-04-20 14:51:54 | INFO     | Visualizer | Visual flawlessly written to: outputs/01_extraction/01_raw_triples_graph.html


### 2. Embedding Compression


In [4]:
embedder = EmbeddingPipeline("config.yaml")

proposed_clusters = embedder.process_triples(raw_triples)
_save_state(proposed_clusters, "outputs/02_embedding/clusters_identified.json")
_dump_current_log()

print(f"Mathematical Compression Output: {len(proposed_clusters)} distinct physical clusters grouped by Euclidean cosine thresholds.")


2026-04-20 14:51:54 | INFO     | EmbeddingPipeline | Initializing Offline Embedding Pipeline (Model: all-MiniLM-L6-v2)
2026-04-20 14:51:54 | INFO     | EmbeddingPipeline | Loading embedding model locally from: models/embeddings/all-MiniLM-L6-v2


This model was created with Sentence Transformers version 5.4.1, but you're using version 5.4.0. Consider updating to the latest version to avoid potential issues.


2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Executing Offline Embedding Matrix completely accurately intelligently.
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Grouping 'subject' logic array accurately cleanly dependably gracefully.
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Processing vector mappings rigidly explicitly cleanly.
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Generating vectors. Unique items: 81
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Dynamic Eigengap Analysis identified optimal components: 31/175
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | PCA reduced logically gracefully. Dimensions elegantly: (81, 31)
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Grouping 'object' logic array accurately cleanly dependably gracefully.
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Processing vector mappings rigidly explicitly cleanly.
2026-04-20 14:51:57 | INFO     | EmbeddingPipeline | Generating vectors. Unique ite

### 3. Taxonomic Hypernym Lifting (Hybrid)


In [5]:
hypernyms = HypernymPipeline("config.yaml")

verified_clusters = await hypernyms.validate_context_vectors(proposed_clusters, master_domain)
_save_state(verified_clusters, "outputs/03_hypernym_lifting/verified_clusters.json")
_dump_current_log()

hypernym_mapping = await hypernyms.taxonomic_lift(verified_clusters, master_domain)
_save_state(hypernym_mapping, "outputs/03_hypernym_lifting/hypernym_mapping.json")
_dump_current_log()

print("Hybrid LLM Verification and taxonomic mapping extracted explicitly.")


2026-04-20 14:51:57 | INFO     | HypernymPipeline | Initializing Master LLM Factory for context lifting.
2026-04-20 14:51:59 | INFO     | ContextManager | Initialized ContextManager - VRAM Free: 11543 MB, RAM Free: 20638 MB
2026-04-20 14:51:59 | INFO     | HypernymPipeline | Initializing Dedicated Embedding Encoder for centroid projections.
2026-04-20 14:51:59 | INFO     | HypernymPipeline | Loading embedding model locally from: models/embeddings/all-MiniLM-L6-v2


This model was created with Sentence Transformers version 5.4.1, but you're using version 5.4.0. Consider updating to the latest version to avoid potential issues.


2026-04-20 14:51:59 | INFO     | HypernymPipeline | Phase 3.1: Validating mathematical proposals...
2026-04-20 14:51:59 | INFO     | HypernymPipeline | Evaluating 74 mathematical clusters for [subject]
2026-04-20 14:52:29 | INFO     | HypernymPipeline | Evaluating 0 mathematical clusters for [predicate]
2026-04-20 14:52:29 | INFO     | HypernymPipeline | Evaluating 158 mathematical clusters for [object]
2026-04-20 14:53:09 | INFO     | HypernymPipeline | Phase 3.2: Native Taxonomic Lifting.
2026-04-20 14:53:45 | ERROR    | SemanticLLMClient | Instructor API Execution failed: <failed_attempts>

<generation number="1">
<exception>
    3 validation errors for TaxonomicVerification
formal_hypernym
  Field required [type=missing, input_value={'description': 'Evaluate...tion', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
excluded_opposite
  Field required [type=missing, input_value={'description': 'Evaluate...tion', 'type': 

### 4. Taxonomic Naming Resolution


In [6]:
mapper = NamingResolutionPipeline()

mapped_triples = mapper.resolve_names(raw_triples, hypernym_mapping)
_save_state(mapped_triples, "outputs/04_mapping/mapped_triplets.json")
_dump_current_log()

visualizer.visualize_triples(mapped_triples, "outputs/04_mapping/02_resolved_triples_graph.html", "Phase 4: Abstracted Taxonomy")
IPython.display.display(IPython.display.IFrame("outputs/04_mapping/02_resolved_triples_graph.html", width="100%", height="600px"))


2026-04-20 14:54:49 | INFO     | NamingResolution | Initializing Taxonomic Resolution & Consolidation Pipeline.
2026-04-20 14:54:49 | INFO     | NamingResolution | Resolving taxonomy natively for 175 semantic triples.
2026-04-20 14:54:49 | INFO     | NamingResolution | Resolution cleanly completed safely. Triples formally updated: 175
2026-04-20 14:54:49 | INFO     | Visualizer | Generating abstract visual organically seamlessly correctly optimally for 175 logical facts securely.
2026-04-20 14:54:49 | INFO     | Visualizer | Visual flawlessly written to: outputs/04_mapping/02_resolved_triples_graph.html


### 5. NetworkX Topological Community Graphing


In [7]:
topology = TopologyEngine()

graph = topology.build_graph(mapped_triples)
partition = topology.detect_communities(graph)
hierarchy = topology.extract_hierarchy(graph, partition)

_save_state(partition, "outputs/05_topology/modularity_partition.json")
_dump_current_log()
_save_state(hierarchy, "outputs/05_topology/extracted_hierarchy.json")
_dump_current_log()

visualizer.visualize_topology(graph, partition, "outputs/05_topology/03_topology_communities_graph.html", "Phase 5: Modularity Topology")
IPython.display.display(IPython.display.IFrame("outputs/05_topology/03_topology_communities_graph.html", width="100%", height="600px"))

# --- Hypergraph Expansion ---
print("Building N-ary Hypergraph Topology matrices...")
hypergraph_res = topology.build_hypergraph_topology(mapped_triples)
topology.visualize_hypergraph(hypergraph_res["B"], "outputs/05_topology")
print("Hypergraph visualization complete.")


2026-04-20 14:54:49 | INFO     | TopologyEngine | Initializing TopologyEngine statically.
2026-04-20 14:54:49 | INFO     | TopologyEngine | Synthesizing structured framework securely from 175 formal semantic triples.
2026-04-20 14:54:49 | INFO     | TopologyEngine | Topological Map completely mapped explicitly structurally natively. Nodes: 237, Edges: 170
2026-04-20 14:54:49 | INFO     | TopologyEngine | Computing Leiden best_partition securely mathematically safely...
2026-04-20 14:54:49 | INFO     | TopologyEngine | Resolution identified securely. Unique functional subsets: 69
2026-04-20 14:54:49 | INFO     | TopologyEngine | Computing explicit global topological relations formally naturally natively.
2026-04-20 14:54:49 | INFO     | Visualizer | Generating geometric topology abstraction logically natively safely.
2026-04-20 14:54:49 | INFO     | Visualizer | Structural Geometric Community visually exported to: outputs/05_topology/03_topology_communities_graph.html


### 6. Semantic JSON Structure Synthesis


In [ ]:
synthesizer = SynthesisEngine("config.yaml")
resolved_schemas = await synthesizer.generate_schemas(hierarchy, master_domain)
file_path = synthesizer.build_global_context(resolved_schemas)

print(f"Final output synthesized seamlessly to: {file_path}")


2026-04-20 14:54:49 | INFO     | SynthesisEngine | Initializing Master LLM Factory for semantic synthesis mappings natively.
2026-04-20 14:54:50 | INFO     | ContextManager | Initialized ContextManager - VRAM Free: 11467 MB, RAM Free: 20613 MB
2026-04-20 14:54:50 | INFO     | SynthesisEngine | Synthesis Engine Initialized natively.
2026-04-20 14:54:50 | INFO     | SynthesisEngine | Synthesizing 69 mapped communities dynamically.


### 7. Logging & Diagnostic Preservation

In [ ]:
_dump_current_log()
